# 📓 Notebook 2 — Model Training & Raw Calibration Measurement
**Uncertainty Quantification in ML Classifiers: A Calibration Benchmarking Study**

---
**Prerequisite:** Run Notebook 1 first.  
If you skipped it, this notebook will auto-generate `splits.pkl` for you.


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os, sys, pickle, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
warnings.filterwarnings('ignore')

from sklearn import datasets as sk_datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
})
PALETTE = {
    'Logistic Regression': '#1d9e75', 'Random Forest':     '#a32d2d',
    'Gradient Boosting':   '#ba7517', 'SVM (RBF)':         '#185fa5',
    'MLP':                 '#533ab7',
}

# ── Path resolution (local Jupyter / VS Code / Google Colab / nbconvert) ─────
DATA_DIR = os.path.join(os.getcwd(), 'data')
os.makedirs(DATA_DIR, exist_ok=True)

def data(filename):
    """Always returns an absolute path to data/<filename> in the working directory."""
    return os.path.join(DATA_DIR, filename)

print(f"Working directory : {os.getcwd()}")
print(f"Data directory    : {DATA_DIR}")
print(f"Python            : {sys.version.split()[0]}")


In [ ]:
# ── Dataset loading ───────────────────────────────────────────────────────────
def load_datasets():
    loaders = {
        'breast_cancer': sk_datasets.load_breast_cancer,
        'wine':          sk_datasets.load_wine,
        'diabetes':      sk_datasets.load_diabetes,
    }
    out = {}
    for name, fn in loaders.items():
        raw = fn()
        X = pd.DataFrame(raw.data, columns=raw.feature_names)
        if name == 'diabetes':
            y = (raw.target > np.median(raw.target)).astype(int)
            tnames = ['low-progression', 'high-progression']
        else:
            y, tnames = raw.target, list(raw.target_names)
        out[name] = {'X': X, 'y': y, 'target_names': tnames}
    return out

# ── Split helper ──────────────────────────────────────────────────────────────
def make_splits(X, y, seed=RANDOM_SEED):
    """60% train | 20% calib | 20% test, stratified."""
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=0.40, random_state=seed, stratify=y)
    X_cal, X_te, y_cal, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=seed, stratify=y_tmp)
    return X_tr, X_cal, X_te, y_tr, y_cal, y_te

def build_and_save_splits():
    """Build splits from scratch and save to data/. Returns splits dict."""
    print("Building splits from scratch...")
    ds = load_datasets()
    splits = {}
    for name, d in ds.items():
        X, y = d['X'].values, d['y']
        X_tr, X_cal, X_te, y_tr, y_cal, y_te = make_splits(X, y)
        sc = StandardScaler().fit(X_tr)
        splits[name] = {
            'X_train': sc.transform(X_tr), 'y_train': y_tr,
            'X_calib': sc.transform(X_cal), 'y_calib': y_cal,
            'X_test':  sc.transform(X_te),  'y_test':  y_te,
        }
    with open(data('splits.pkl'), 'wb') as f: pickle.dump(splits, f)
    with open(data('meta.pkl'),   'wb') as f:
        pickle.dump({k: {'target_names': v['target_names']} for k,v in ds.items()}, f)
    print(f"  Saved {data('splits.pkl')}")
    return splits

def load_splits():
    """Load splits, auto-building them if missing."""
    p = data('splits.pkl')
    if not os.path.exists(p):
        print(f"splits.pkl not found at {p}")
        print("Run Notebook 1 first, or auto-generating now...")
        return build_and_save_splits()
    with open(p, 'rb') as f:
        return pickle.load(f)

# ── Classifiers ───────────────────────────────────────────────────────────────
def get_classifiers():
    return {
        'Logistic Regression': LogisticRegression(
            C=1.0, max_iter=1000, random_state=RANDOM_SEED),
        'Random Forest':       RandomForestClassifier(
            n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
        'Gradient Boosting':   GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.1, random_state=RANDOM_SEED),
        'SVM (RBF)':           SVC(
            kernel='rbf', C=1.0, gamma='scale',
            probability=True, random_state=RANDOM_SEED),
        'MLP':                 MLPClassifier(
            hidden_layer_sizes=(100, 50), activation='relu',
            max_iter=500, random_state=RANDOM_SEED),
    }

# ── Calibration metrics ───────────────────────────────────────────────────────
def compute_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece, n = 0.0, len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (y_prob >= lo) & (y_prob < hi)
        if not m.any(): continue
        ece += m.sum() / n * abs(float(y_true[m].mean()) - float(y_prob[m].mean()))
    return ece

def compute_mce(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1); mce = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (y_prob >= lo) & (y_prob < hi)
        if not m.any(): continue
        mce = max(mce, abs(float(y_true[m].mean()) - float(y_prob[m].mean())))
    return mce

def brier_score(y_true, y_prob):
    return float(np.mean((np.array(y_true) - np.array(y_prob)) ** 2))

def reliability_data(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    centers, confs, freqs, sizes = [], [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (y_prob >= lo) & (y_prob < hi)
        if not m.any(): continue
        centers.append((lo+hi)/2); confs.append(float(y_prob[m].mean()))
        freqs.append(float(y_true[m].mean())); sizes.append(int(m.sum()))
    return np.array(centers), np.array(confs), np.array(freqs), np.array(sizes)

def get_probs(clf, X, y):
    """Binary → prob of positive class; multi-class → top-label ECE."""
    proba = clf.predict_proba(X)
    if proba.shape[1] == 2:
        return np.array(y), proba[:, 1]
    return (clf.predict(X) == np.array(y)).astype(int), proba.max(axis=1)

# ── Calibrators ───────────────────────────────────────────────────────────────
class PlattScaler:
    def __init__(self):
        self.lr = LogisticRegression(C=1.0, solver='lbfgs')
        self._fallback = False
    def fit(self, p, y):
        if len(set(map(int, y))) < 2:
            self._fallback = True
        else:
            self._fallback = False
            self.lr.fit(p.reshape(-1, 1), y)
        return self
    def predict_proba(self, p):
        return p if self._fallback else self.lr.predict_proba(p.reshape(-1,1))[:,1]

class IsotonicScaler:
    def __init__(self):
        self.ir = IsotonicRegression(out_of_bounds='clip')
        self._fallback = False
    def fit(self, p, y):
        if len(set(map(int, y))) < 2:
            self._fallback = True
        else:
            self._fallback = False
            self.ir.fit(p, y)
        return self
    def predict_proba(self, p):
        return p if self._fallback else self.ir.predict(p)

print("✓ All utilities loaded")


## 1 · Load Splits

In [ ]:
splits = load_splits()   # auto-generates if splits.pkl is missing
DATASET_NAMES = list(splits.keys())
MODEL_NAMES   = list(get_classifiers().keys())
print("Datasets:", DATASET_NAMES)
print("Models  :", MODEL_NAMES)


## 2 · Train All Models

In [ ]:
trained_models = {}
raw_probs      = {}
raw_metrics    = []

for ds_name in DATASET_NAMES:
    sp = splits[ds_name]
    trained_models[ds_name] = {}
    raw_probs[ds_name]      = {}
    print(f"\n{'═'*52}\n  {ds_name.upper()}\n{'═'*52}")

    for clf_name, clf in get_classifiers().items():
        t0 = time.time()
        clf.fit(sp['X_train'], sp['y_train'])
        elapsed = time.time() - t0

        y_eval, prob_pos = get_probs(clf, sp['X_test'], sp['y_test'])
        acc   = accuracy_score(sp['y_test'], clf.predict(sp['X_test']))
        ece   = compute_ece(y_eval, prob_pos)
        mce   = compute_mce(y_eval, prob_pos)
        brier = brier_score(y_eval, prob_pos)

        trained_models[ds_name][clf_name] = clf
        raw_probs[ds_name][clf_name]      = (y_eval, prob_pos)
        raw_metrics.append({'Dataset': ds_name, 'Model': clf_name,
                             'Accuracy': round(acc,4), 'ECE': round(ece,4),
                             'MCE': round(mce,4), 'Brier': round(brier,4),
                             'Time_s': round(elapsed,2)})
        print(f"  {clf_name:<22} Acc={acc:.3f} ECE={ece:.4f} MCE={mce:.4f} ({elapsed:.1f}s)")

raw_metrics_df = pd.DataFrame(raw_metrics)
print("\n✓ Training complete")


## 3 · Results Matrix

In [ ]:
pivot = raw_metrics_df.pivot_table(
    index='Model', columns='Dataset',
    values=['Accuracy','ECE','MCE','Brier'], aggfunc='mean').round(4)
print("Mean metrics across datasets:")
display(pivot)

ece_sum = (raw_metrics_df.groupby('Model')['ECE']
           .agg(['mean','std']).round(4)
           .rename(columns={'mean':'Mean ECE','std':'Std ECE'})
           .sort_values('Mean ECE'))
print("\nECE Summary:")
display(ece_sum)


## 4 · Reliability Diagrams

In [ ]:
ds = 'breast_cancer'
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)
fig.suptitle(f'Reliability Diagrams — Breast Cancer (diagonal = perfect calibration)',
             fontweight='bold', y=1.02)

for ax, clf_name in zip(axes, MODEL_NAMES):
    y_eval, prob_pos = raw_probs[ds][clf_name]
    _, confs, freqs, _ = reliability_data(y_eval, prob_pos)
    col = PALETTE[clf_name]
    ax.bar(confs, np.abs(freqs - confs), bottom=np.minimum(confs, freqs),
           width=0.08, color=col, alpha=0.25)
    ax.plot(confs, freqs, 'o-', color=col, ms=6, lw=2)
    ax.plot([0,1],[0,1], '--', color='#aaa', lw=1.2, zorder=0)
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_aspect('equal')
    ax.set_xlabel('Mean Confidence')
    if ax == axes[0]: ax.set_ylabel('Observed Frequency')
    ax.set_title(f'{clf_name}\nECE={compute_ece(y_eval,prob_pos):.4f}', fontsize=9)

plt.tight_layout()
plt.savefig('fig_reliability_diagrams.png', bbox_inches='tight', dpi=150)
plt.show()
print("→ Saved fig_reliability_diagrams.png")


## 5 · ECE Comparison Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.suptitle('ECE by Model and Dataset (lower = better)', fontweight='bold')

for ax, ds_name in zip(axes, DATASET_NAMES):
    sub = raw_metrics_df[raw_metrics_df['Dataset']==ds_name].sort_values('ECE')
    bars = ax.barh(sub['Model'], sub['ECE'],
                   color=[PALETTE[m] for m in sub['Model']],
                   edgecolor='white', height=0.55)
    ax.axvline(0.05, color='#1d9e75', ls=':', lw=1.5, label='ECE=0.05')
    ax.set_xlabel('ECE')
    ax.set_title(ds_name.replace('_',' ').title())
    for bar, val in zip(bars, sub['ECE']):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)
    if ax == axes[0]: ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig_ece_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print("→ Saved fig_ece_comparison.png")


## 6 · Save Artefacts

In [ ]:
with open(data('trained_models.pkl'), 'wb') as f: pickle.dump(trained_models, f)
with open(data('raw_probs.pkl'),      'wb') as f: pickle.dump(raw_probs, f)
with open(data('raw_metrics.pkl'),    'wb') as f: pickle.dump(raw_metrics_df, f)

lr_ece = raw_metrics_df[raw_metrics_df['Model']=='Logistic Regression']['ECE'].mean()
rf_ece = raw_metrics_df[raw_metrics_df['Model']=='Random Forest']['ECE'].mean()
print(f"Best calibrated : Logistic Regression (mean ECE {lr_ece:.4f})")
print(f"Worst calibrated: Random Forest       (mean ECE {rf_ece:.4f})")
print(f"Overconfidence gap: +{(rf_ece-lr_ece)*100:.0f}pp above LR baseline")
print("\n✓ Notebook 2 complete — run Notebook 3 next")
